# 03 — Unicycle Data + Diffusion Training

This notebook generates unicycle trajectories, trains a diffusion score model, and evaluates sampled trajectories against true dynamics.

### Setup

Install dependencies, set seeds, and define paths for data/config/output.

In [ ]:
from pathlib import Path
import os
import random
import subprocess
import sys

import numpy as np
import torch

# If the repo is not present in /content, set REPO_URL to your GitHub repo and rerun.
REPO_URL = "https://github.com/<your-org>/score-manifold-optimization.git"
REPO_DIR = Path("/content/score-manifold-optimization")

if not (REPO_DIR / "pyproject.toml").exists():
    if "<" in REPO_URL:
        raise RuntimeError(
            "Repo not found at /content/score-manifold-optimization. "
            "Please set REPO_URL to your repository URL or clone manually."
        )
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = REPO_DIR / "data"
OUTPUT_DIR = REPO_DIR / "outputs/unicycle_train_demo"
CONFIG_PATH = REPO_DIR / "configs/train_unicycle_quick.yaml"
DATASET_PATH = DATA_DIR / "unicycle_T100_n20000.pt"

print(f"Repo: {REPO_DIR}")
print(f"Device: {DEVICE}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Config path: {CONFIG_PATH}")
print(f"Output dir: {OUTPUT_DIR}")


### Local Setup (Alternative)

If the repository is already cloned locally, run this cell instead of the setup cell above.

In [ ]:
from pathlib import Path
import os
import random

import numpy as np
import torch

start = Path.cwd().resolve()
REPO_DIR = next((p for p in [start, *start.parents] if (p / "pyproject.toml").exists()), None)
if REPO_DIR is None:
    raise RuntimeError("Could not find repo root (missing pyproject.toml in parent dirs).")

os.chdir(REPO_DIR)

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = REPO_DIR / "data"
OUTPUT_DIR = REPO_DIR / "outputs/unicycle_train_demo"
CONFIG_PATH = REPO_DIR / "configs/train_unicycle_quick.yaml"
DATASET_PATH = DATA_DIR / "unicycle_T100_n20000.pt"

print(f"Repo root: {REPO_DIR}")
print(f"Device: {DEVICE}")
print(f"Dataset path: {DATASET_PATH}")
print(f"Config path: {CONFIG_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

# Install once from terminal (repo root): pip install -e .


### Generate Data

Generate 20,000 unicycle trajectories of length 100 with x0 = 0 and bounded control inputs.

In [ ]:
from datetime import datetime
from diffusion.control import Unicycle
import matplotlib.pyplot as plt


DATA_DIR.mkdir(parents=True, exist_ok=True)

HORIZON = 100
N_TRAIN = 20000
N_TEST = 0
N_VAL = 0
system = Unicycle(dT=0.05)

u = torch.empty(N_TRAIN, HORIZON, system.get_input_dim(), dtype=torch.float32)
u[..., 0].uniform_(-10.0, 10.0)  # tangential velocity input
u[..., 1].uniform_(-5.0, 5.0)    # angle input

# Here we always start at the origin
x0 = torch.zeros(N_TRAIN, system.get_state_dim(), dtype=torch.float32)

with torch.no_grad():
    y = system.simulate_out(x0, u)

train_data = torch.cat([u, y], dim=-1)
empty_split = torch.empty((0, HORIZON, train_data.shape[-1]), dtype=train_data.dtype)

metadata = {
    "space": {
        "class": "TrajectorySpace",
        "params": {
            "horizon": HORIZON,
            "input_dim": int(system.get_input_dim()),
            "output_dim": int(system.get_output_dim()),
        },
    },
    "constraint": {
        "class": "Unicycle",
        "params": {"dT": float(system.dT)},
    },
    "state_dim": int(system.get_state_dim()),
    "n_train": int(N_TRAIN),
    "n_test": int(N_TEST),
    "n_val": int(N_VAL),
    "created": datetime.now().isoformat(),
    "version": "1.0",
    "generation_config": {
        "horizon": HORIZON,
        "input_range": {"tangential_velocity": [-10.0, 10.0], "angle_input": [-5.0, 5.0]},
        "x0": "zeros",
    },
}

dataset = {
    "train_data": train_data,
    "test_data": empty_split,
    "val_data": empty_split.clone(),
    "metadata": metadata,
}
torch.save(dataset, DATASET_PATH)

print(f"Train data shape: {tuple(dataset['train_data'].shape)}")
print(f"Saved dataset: {DATASET_PATH}")

#Plot some of the trajectories 
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
N_PLOT = 100
for j in range(N_PLOT):
    axes[0].plot(dataset["train_data"][j, :, 2], dataset["train_data"][j, :, 3]) #Only x-y coordinates

for j in range(N_PLOT):
    axes[1].plot(dataset["train_data"][j, :, 0:2]) #Output


### Train Model

Train directly through the Python API and save canonical checkpoint artifacts.

Note that depending on the device this can take quite some time (up to multiple hours)

In [ ]:
from datetime import datetime
import json, yaml

from diffusion.data import load_dataset
from diffusion.models import create_model
from diffusion.training import DiffusionTrainer, TrainingOptions, create_diffusion, create_optimizer

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

dataset = load_dataset(str(DATASET_PATH), device=DEVICE)
space = dataset.get_space()
diffusion = create_diffusion(space, cfg)
model = create_model(
    model_type=str(cfg["model"]["type"]),
    space=space,
    diffusion=diffusion,
    config=cfg,
    initialize=True,
).to(DEVICE)
optimizer = create_optimizer(model, cfg)

options_dict = dict(cfg.get("training_options", {}))
options_dict["batch_size"] = int(cfg.get("training", {}).get("batch_size", 32))
options = TrainingOptions(**options_dict)

constraint = dataset.constraint
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

def log_fn(samples):
    violations = constraint.violation(samples)
    return {
        "constraint_violation_mean": violations.mean().item(),
        "constraint_violation_std": violations.std().item(),
        "constraint_violation_max": violations.max().item(),
    }

trainer = DiffusionTrainer(
    diffusion=diffusion,
    score_model=model,
    train_data=dataset.train_data.to(DEVICE),
    optimizer=optimizer,
    options=options,
    log_fn=log_fn,
    device=DEVICE,
    output_dir=OUTPUT_DIR,
    metadata=dataset.metadata,
)

num_epochs = int(cfg.get("training", {}).get("num_epochs", 80))
trainer.train(num_epochs=num_epochs)
trainer.save_checkpoint(str(OUTPUT_DIR / "checkpoint.pt"))

with open(OUTPUT_DIR / "config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

summary = {
    "timestamp": datetime.now().isoformat(),
    "output_dir": str(OUTPUT_DIR),
    "num_epochs": num_epochs,
    "final_loss": float(trainer.train_losses[-1]) if trainer.train_losses else None,
    "num_params": num_params,
    "dataset_path": str(DATASET_PATH),
    "model_type": str(cfg["model"]["type"]),
}
with open(OUTPUT_DIR / "train_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Training complete. Artifacts written to {OUTPUT_DIR}")


### Verify Outputs

Check that training artifacts were written and print the summary.

In [ ]:
import json

required = ["model.pth", "checkpoint_data.pt", "config.yaml", "metadata.json", "train_summary.json"]
for name in required:
    path = OUTPUT_DIR / name
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

with open(OUTPUT_DIR / "train_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)

print("\nTrain summary:")
print(json.dumps(summary, indent=2))


### Sample + Dynamics Error

Sample trajectories from the trained model and compare generated outputs against true unicycle rollouts.

In [ ]:
N_SAMPLES = 4


import yaml
from diffusion.data import load_dataset
from diffusion.training import create_diffusion
from diffusion.models import create_model


with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
print(cfg)
print(DATASET_PATH)

dataset = load_dataset(str(DATASET_PATH), device=DEVICE)
space = dataset.get_space()
constraint = dataset.constraint
diffusion = create_diffusion(space, cfg)

model = create_model(
    model_type=str(cfg["model"]["type"]),
    space=space,
    diffusion=diffusion,
    config=cfg,
    initialize=False,
).to(DEVICE)

state_dict = torch.load(OUTPUT_DIR / "model.pth", map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

with torch.no_grad():
    samples = diffusion.sample_reverse(
        N_sample=N_SAMPLES,
        score_model=model,
        flow_type="ODE",
        end_only=True,
        device=DEVICE,
    )

u, y_gen = constraint.split_input_output(samples)
system = constraint.system
x0_zero = torch.zeros(N_SAMPLES, system.get_state_dim(), device=samples.device, dtype=samples.dtype)

with torch.no_grad():
    y_true = system.simulate_out(x0_zero, u)

err = y_gen - y_true
mae = err.abs().mean(dim=(1, 2)).detach().cpu()
rmse = err.pow(2).mean(dim=(1, 2)).sqrt().detach().cpu()
violation = constraint.violation(samples).detach().cpu()

print(f"Samples shape: {tuple(samples.shape)}")
print(
    "Output error (generated vs true) | "
    f"MAE mean={mae.mean().item():.4e}, median={mae.median().item():.4e}, max={mae.max().item():.4e}"
)
print(
    "Output error (generated vs true) | "
    f"RMSE mean={rmse.mean().item():.4e}, median={rmse.median().item():.4e}, max={rmse.max().item():.4e}"
)
print(
    "Dynamics violation | "
    f"mean={violation.mean().item():.4e}, median={violation.median().item():.4e}, max={violation.max().item():.4e}"
)


### Plot Example Trajectories

Plot a few generated vs true trajectories in the x-y plane.

In [ ]:
import matplotlib.pyplot as plt
num_plot = min(3, N_SAMPLES)

fig, axes = plt.subplots(1, num_plot, figsize=(5 * num_plot, 4), squeeze=False)

for i in range(num_plot):
    ax = axes[0, i]
    ax.plot(y_true[i, :, 0].detach().cpu(), y_true[i, :, 1].detach().cpu(), "k--", label="True")
    ax.plot(y_gen[i, :, 0].detach().cpu(), y_gen[i, :, 1].detach().cpu(), color="tab:blue", label="Generated")
    ax.set_title(f"Trajectory {i}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()
